# Cross-validation sweep — BanglaPoliticalStance

Runs every model in `configs/` under **one** protocol: grouped stratified 5-fold CV,
augmentation inside training folds, out-of-fold predictions pooled.

**Setup:** Runtime → Change runtime type → **T4 GPU**

**6 cells, run them in order. No tokens needed.**

In [ ]:
# Cell 1: GPU check
!nvidia-smi -L
import torch; print(f'torch {torch.__version__}, CUDA {torch.cuda.is_available()}')

In [ ]:
# Cell 2: Clone + install (repo is public)
import os, sys, subprocess, shutil
REPO = '/content/bangla-multimodal-political-stance'
os.chdir('/content')
if os.path.exists(REPO):
    shutil.rmtree(REPO)
subprocess.run(['git', 'clone', '--depth', '1',
    'https://github.com/kishormorol/bangla-multimodal-political-stance.git', REPO], check=True)
os.chdir(REPO)

# Install with full output so we can see errors
!pip install -e . 2>&1 | tail -10
!pip install gdown datasets 2>&1 | tail -3

# Verify
!python -c "import bmpb; print('bmpb installed OK')"
print(f'Working directory: {os.getcwd()}')

In [ ]:
# Cell 3: Build corpus.csv (try Drive first, fall back to HF dataset)
import os, subprocess, sys, time
from pathlib import Path

REPO = '/content/bangla-multimodal-political-stance'
os.chdir(REPO)

# Try downloading from Drive (with retries)
for attempt in range(1, 4):
    print(f'=== bmpb data attempt {attempt}/3 ===')
    subprocess.run([sys.executable, '-m', 'bmpb.cli', 'data'])
    csvs = list(Path(REPO, 'data', 'raw').glob('*.csv'))
    if len(csvs) >= 20:
        print(f'Got {len(csvs)} CSVs.')
        break
    print(f'Got {len(csvs)} CSVs, retrying in 20s...')
    time.sleep(20)

# Try bmpb ingest
result = subprocess.run([sys.executable, '-m', 'bmpb.cli', 'ingest'],
                        capture_output=True, text=True)
print(result.stdout)

# Fallback: build corpus.csv from HF if ingest failed
corpus_path = Path(REPO, 'data', 'processed', 'corpus.csv')
if not corpus_path.exists():
    print('Ingest failed - building corpus.csv from HF dataset...')
    from datasets import load_dataset
    import pandas as pd

    # Load the dataset (image decoding only happens on __getitem__, not here)
    ds = load_dataset('kishormorol/BanglaPoliticalStance', split='annotated')
    LABEL_NAMES = {0: 'govt_critique', 1: 'neutral', 2: 'govt_leaning'}

    # Remove image column so row access never triggers PIL decoding
    ds_text = ds.remove_columns(['image'])

    # Phase 1: build corpus rows from text-only data
    print(f'Building corpus from {len(ds_text)} items (text only)...')
    rows = []
    for idx in range(len(ds_text)):
        item = ds_text[idx]
        rows.append({
            'item_id': item['item_id'],
            'title': item['headline'],
            'text': item['headline'],
            'label': item['label'],
            'label_name': LABEL_NAMES.get(item['label'], 'unknown'),
            'article_label_name': '',
            'image_label_name': '',
            'outlet': item['outlet'],
            'outlet_key': item['outlet'].lower(),
            'date': item['date'],
            'source_url': item['source_url'],
            'image_url': '',
            'image_path': '',
            'image_kind': '',
            'has_image': False,
            'annotator_1': '',
            'annotator_2': '',
            'annotator_3': '',
            'article_label': '',
            'image_label': '',
            'text_level': 'headline',
            'source_index': item['item_id'],
        })
    print(f'  Text corpus built: {len(rows)} items')

    # Phase 2: extract images from the Arrow table (raw bytes, bypasses PIL)
    images_dir = Path(REPO, 'data', 'raw', 'processed_images')
    images_dir.mkdir(parents=True, exist_ok=True)
    n_saved = 0
    n_failed = 0

    try:
        arrow_table = ds._data  # underlying pyarrow Table with raw bytes
        img_col = arrow_table.column('image')
        print(f'Extracting images from Arrow table ({len(img_col)} rows)...')

        import io
        from PIL import Image as PILImage

        for idx in range(len(img_col)):
            item_id = rows[idx]['item_id']
            try:
                img_struct = img_col[idx].as_py()
                if img_struct and img_struct.get('bytes'):
                    raw_bytes = img_struct['bytes']
                    pil_img = PILImage.open(io.BytesIO(raw_bytes))
                    img_filename = item_id + '.jpg'
                    img_save_path = images_dir / img_filename
                    pil_img.convert('RGB').save(img_save_path, format='JPEG', quality=95)
                    rows[idx]['image_path'] = 'data/raw/processed_images/' + img_filename
                    rows[idx]['image_kind'] = 'photo'
                    rows[idx]['has_image'] = True
                    n_saved += 1
            except Exception as e:
                n_failed += 1
                if n_failed <= 5:
                    print(f'  Skipping image for {item_id}: {e}')
    except Exception as e:
        print(f'Could not extract images: {e}')
        print('Continuing with text-only corpus (multimodal CV will be skipped).')

    corpus = pd.DataFrame(rows)
    corpus_path.parent.mkdir(parents=True, exist_ok=True)
    corpus.to_csv(corpus_path, index=False)
    print(f'Created corpus.csv: {len(corpus)} items ({n_saved} images, {n_failed} failed)')

# Verify
import pandas as pd
df = pd.read_csv(corpus_path)
print(f'Corpus ready: {len(df)} items')
print(f'Labels: {df["label_name"].value_counts().to_dict()}')
print(f'Items with images: {int(df["has_image"].sum())}')

In [ ]:
# Cell 4: Run text model CV sweep
import os
os.chdir('/content/bangla-multimodal-political-stance')
!PYTHON=$(which python) bash scripts/run_cv.sh configs/text

In [ ]:
# Cell 5: Run multimodal model CV sweep
import os
os.chdir('/content/bangla-multimodal-political-stance')
!PYTHON=$(which python) bash scripts/run_cv.sh configs/multimodal

In [ ]:
# Cell 6: Leaderboard + download
import os
os.chdir('/content/bangla-multimodal-political-stance')
!python -m bmpb.cli leaderboard
print(open('reports/tables/leaderboard.md').read())

!tar czf /content/cv-runs.tar.gz experiments reports
from google.colab import files
files.download('/content/cv-runs.tar.gz')